# Emotion Detection — Training Notebook
Same stack as the original project: pandas, NLTK preprocessing, TF-IDF, Logistic Regression.

This notebook trains the model and saves `model.pkl`, `vectorizer.pkl`, and `label_map.pkl` for the Streamlit app.

In [1]:
import string
import pickle

import pandas as pd
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

## 1. Load data

In [2]:
df = pd.read_csv('train.txt', sep=';', header=None, names=['text', 'emotion'])
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [3]:
df.isnull().sum()

text       0
emotion    0
dtype: int64

## 2. Encode emotion labels

In [4]:
unique_emotions = df['emotion'].unique()
emotion_numbers = {}
i = 0
for emo in unique_emotions:
    emotion_numbers[emo] = i
    i += 1

df['emotion_label'] = df['emotion'].map(emotion_numbers)
number_to_emotion = {v: k for k, v in emotion_numbers.items()}
df.head()

,text,emotion,emotion_label
0,i didnt feel humiliated,sadness,0
1,i can go from feeling so hopeless to so damned...,sadness,0
2,im grabbing a minute to post i feel greedy wrong,anger,1
3,i am ever feeling nostalgic about the fireplac...,love,2
4,i am feeling grouchy,anger,1


## 3. Text cleaning
Lowercase → remove punctuation → remove numbers → remove non-ASCII/emojis → remove stopwords.

In [5]:
df['text'] = df['text'].apply(lambda x: x.lower())

def remove_punc(txt):
    return txt.translate(str.maketrans('', '', string.punctuation))

df['text'] = df['text'].apply(remove_punc)

def remove_numbers(txt):
    new = ""
    for i in txt:
        if not i.isdigit():
            new = new + i
    return new

df['text'] = df['text'].apply(remove_numbers)

def remove_emojis(txt):
    new = ""
    for i in txt:
        if i.isascii():
            new += i
    return new

df['text'] = df['text'].apply(remove_emojis)

In [6]:
stop_words = set(stopwords.words('english'))

def remove(txt):
    words = txt.split()
    cleaned = []
    for i in words:
        if not i in stop_words:
            cleaned.append(i)
    return ' '.join(cleaned)

df['text'] = df['text'].apply(remove)
df.head()

,text,emotion,emotion_label
0,didnt feel humiliated,sadness,0
1,go feeling hopeless damned hopeful around some...,sadness,0
2,im grabbing minute post feel greedy wrong,anger,1
3,ever feeling nostalgic fireplace know still pr...,love,2
4,feeling grouchy,anger,1


## 4. Train/test split

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['emotion_label'], test_size=0.20, random_state=42
)

## 5. Bag-of-Words + Naive Bayes (baseline)

In [8]:
bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

nb_model = MultinomialNB()
nb_model.fit(X_train_bow, y_train)

pred_bow = nb_model.predict(X_test_bow)
print(accuracy_score(y_test, pred_bow))

0.768125


## 6. TF-IDF + Naive Bayes

In [9]:
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

nb2_model = MultinomialNB()
nb2_model.fit(X_train_tfidf, y_train)

y_pred = nb2_model.predict(X_test_tfidf)
print(accuracy_score(y_test, y_pred))

0.6609375


## 7. TF-IDF + Logistic Regression (best model — used in the deployed app)

In [10]:
logistic_model = LogisticRegression(max_iter=1000)
logistic_model.fit(X_train_tfidf, y_train)

log_pred = logistic_model.predict(X_test_tfidf)
print(accuracy_score(y_test, log_pred))
print(classification_report(y_test, log_pred, target_names=[number_to_emotion[i] for i in sorted(number_to_emotion)]))

0.8628125
              precision    recall  f1-score   support

     sadness       0.90      0.94      0.92       946
       anger       0.90      0.81      0.86       427
        love       0.90      0.61      0.73       296
    surprise       0.88      0.47      0.61       113
        fear       0.86      0.76      0.81       397
         joy       0.81      0.96      0.88      1021

    accuracy                           0.86      3200
   macro avg       0.88      0.76      0.80      3200
weighted avg       0.87      0.86      0.86      3200



## 8. Save artifacts for the Streamlit app

In [11]:
with open('model.pkl', 'wb') as f:
    pickle.dump(logistic_model, f)

with open('vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf_vectorizer, f)

with open('label_map.pkl', 'wb') as f:
    pickle.dump(number_to_emotion, f)

print("Saved model.pkl, vectorizer.pkl, label_map.pkl")

Saved model.pkl, vectorizer.pkl, label_map.pkl


## 9. Quick manual test

In [12]:
def clean_text(txt):
    txt = txt.lower()
    txt = remove_punc(txt)
    txt = remove_numbers(txt)
    txt = remove_emojis(txt)
    txt = remove(txt)
    return txt

sample = "I am so happy today, this is the best day ever!"
cleaned = clean_text(sample)
vec = tfidf_vectorizer.transform([cleaned])
pred = logistic_model.predict(vec)[0]
print(sample, '->', number_to_emotion[pred])

I am so happy today, this is the best day ever! -> joy
